# Exploring a brewing yeast genome — `FLO11` variant graph

This notebook analyses the *FLO11* locus using a 9.1 kb fragment of
*S. cerevisiae* chrIX and serves as both a biological showcase and a
complete tutorial for the `gen` Python widget API.

| Gene | Function | Brewing relevance |
|------|----------|-------------------|
| **FLO11** (*MUC1/YIR019C*) | GPI-anchored cell-surface flocculin | Pseudohyphal growth, flocculation, biofilm |

FLO11 is transcriptionally controlled by two converging MAPK pathways
via a composite *Filamentation Response Element* (FRE) in its promoter:

| Element | TF | Canonical sequence | Pathway |
|---------|----|--------------------|----------|
| **PRE** | Ste12 | `TGAAACA` | Filamentous growth |
| **TCS** | Tec1 | `CATTCC` / `CATTCT` | Filamentous growth (co-activator) |
| **STRE** | Msn2/Msn4 | `AGGGG` | Stress response (competitor) |
| **TATA box** | TBP | `TATAAA` | Core promoter |

**BRQ** is a brewing yeast strain. We load the S288c reference sequence
and apply BRQ variant calls to build a variant graph encoding both haplotypes.

**FLO11 orientation note** — FLO11 is on the **minus strand**, occupying
positions 1–4104 of this fragment. The large intergenic region to the
right (~4200 onward) is the promoter. Regulatory elements on the minus strand
(`strand == "-"`) in the promoter region are the ones driving FLO11 expression.

---
## Part 1 — Setup & first look

**API covered:** `bg.plot()`, `fig.show_path()`, `fig.add_annotation_track_file()`,
`fig.annotation_tracks()`

### Setup

Fixture files (`flo11_reference.fa`, `flo11_variants.vcf`, `flo11_reference.gff3`)
live alongside this notebook. A fresh temporary repository is created each run
so repeated executions do not accumulate state.

In [1]:
import pathlib
import tempfile

import gen

EXAMPLES_DIR = pathlib.Path(".").resolve()
FASTA = EXAMPLES_DIR / "flo11_reference.fa"
VCF   = EXAMPLES_DIR / "flo11_variants.vcf"
GFF3  = EXAMPLES_DIR / "flo11_reference.gff3"

assert FASTA.exists(), f"Missing: {FASTA}"
assert VCF.exists(),   f"Missing: {VCF}"
assert GFF3.exists(),  f"Missing: {GFF3}"

WORK_DIR = pathlib.Path(tempfile.mkdtemp(prefix="gen-flo11-"))
print(f"Working in {WORK_DIR}")

repo = gen.Repository(str(WORK_DIR))
repo.import_fasta(str(FASTA), sample='reference')
repo.update_with_vcf(str(VCF), reference='reference')  # creates BRQ sample

bgs = repo.get_sequence_graphs()
print(f"{len(bgs)} block group(s) imported")
for b in bgs:
    print(f"  {b.sample_name:15s}  {b.name}")

bg_brq = next(b for b in bgs if b.sample_name == 'BRQ')
bg_ref = next(b for b in bgs if b.sample_name == 'reference')

Working in /var/folders/f8/8zf8xczs0pxf9_vfnlmqlx9h0000gn/T/gen-flo11-4r0p2dh1
2 block group(s) imported
  reference        pad1_fdc1_region
  BRQ              pad1_fdc1_region


### Plot the BRQ variant graph

`bg.plot()` renders the block group as an interactive graph widget.
Nodes shared between haplotypes appear once; divergent nodes form bubbles.

`fig.show_path()` draws the BRQ haplotype path as a coloured ribbon over the
graph topology, making it easy to trace the variant path through shared nodes.

In [2]:
fig_path = bg_brq.plot()
fig_path.show_path()
fig_path

### Gene annotations from GFF3

`add_annotation_track_file()` projects GFF3 features onto the graph as
coloured bars in a track panel below the canvas. We filter to `gene` rows
to show FLO11 and flanking genes without redundant CDS / chromosome entries.

With gene boundaries visible, positions in subsequent search results immediately
make biological sense: left of position ~4104 is coding sequence; right is promoter.

In [3]:
fig_genes = bg_brq.plot(rows=24)

fig_genes.add_annotation_track(
    file=str(GFF3),
    name="genes",
)

print("Track panels:", fig_genes.annotation_tracks())
fig_genes

Track panels: ['genes']


In [4]:
fig_genes = bg_brq.plot(rows=24)

fig_genes.add_annotation_track(
    file=str(GFF3),
    filter=lambda row: row.split("\t")[2] == "gene",
    name="genes",
)

print("Track panels:", fig_genes.annotation_tracks())
fig_genes

Track panels: ['genes']


---
## Part 2 — Searching without an index

**API covered:** `bg.search()`, `bg.get_locus_sequence()`, `fig.add_annotation()`,
`fig.go_to()`, `fig.show()`, `fig.inline_annotations()`

`bg.search()` works immediately — no index required. Without a pre-built index
it performs a full graph scan. DNA mode is case-insensitive and returns hits on
**both strands** in a single call. Use `h.strand` (`"+"` or `"-"`) to
distinguish orientation afterward.

### TATA box — core promoter anchor

The TATA box (`TATAAA`) sits ~25–30 bp upstream of a transcription start site.
For a minus-strand gene like FLO11, the TATA box is in the promoter region
to the right of the coding sequence (`strand == "-"` hits near position 4104).

We call `bg.get_locus_sequence()` on the first few hits to confirm the search
is returning the expected sequence — useful when validating novel motifs.

In [5]:
tata_hits = bg_brq.search("tataaa")

tata_fwd = [h for h in tata_hits if h.strand == "+"]
tata_rev = [h for h in tata_hits if h.strand == "-"]
print(f"TATA hits  forward (+): {len(tata_fwd)}")
print(f"TATA hits  reverse (-): {len(tata_rev)}")
print(f"TATA hits  total:       {len(tata_hits)}")
print()

# get_locus_sequence() not available — print hit positions instead
print("Sampled hits (expect TATAAA or reverse complement TTTATA):")
for h in tata_hits[:4]:
    print(f"  strand={h.strand}  start={h.start}  end={h.end}")

AttributeError: 'builtins.GraphLocus' object has no attribute 'strand'

### Inline annotations and navigation

`fig.add_annotation()` renders an annotation **directly on the graph canvas** —
each hit is tinted with an accent colour and labelled just below its span.
This is ideal for a focused set of important sites.

`fig.go_to()` pans the viewport to a graph position.
`fig.inline_annotations()` returns the names of all current inline annotations.

In [ ]:
fig_tata_inline = bg_brq.plot(rows=28)
fig_tata_inline.add_annotation_track(
    file=str(GFF3),
    filter=lambda row: row.split("\t")[2] == "gene",
    name="genes",
)

for i, h in enumerate(tata_hits):
    fig_tata_inline.add_annotation(gen.Annotation(h, name=f"TATA{h.strand}:{i}"))

print("Inline annotations:", fig_tata_inline.inline_annotations())

# Navigate to the first reverse-strand TATA hit — those sit in the FLO11 promoter
ref_tata = tata_rev[0] if tata_rev else tata_hits[0]
fig_tata_inline.go_to(ref_tata.start())

fig_tata_inline

---
## Part 3 — Annotation layers

**API covered:** `fig.add_annotation_track()`, `fig.remove_annotation_track()`,
`fig.annotation_tracks()`, `fig.inline_annotations()`

### PRE and TCS sites

The FLO11 promoter contains multiple Ste12 PRE (`TGAAACA`) and Tec1 TCS sites
(`CATTCC`, `CATTCT`). When present within ~100 bp of each other, PRE + TCS form
a *Filamentation Response Element* that drives pseudohyphal growth.

A single DNA-mode search per motif returns all hits on both strands.

In [ ]:
pre_hits  = bg_brq.search("tgaaaca")
tcs1_hits = bg_brq.search("cattcc")
tcs2_hits = bg_brq.search("cattct")
tcs_hits  = tcs1_hits + tcs2_hits

for label, hits in [("PRE (Ste12)", pre_hits),
                    ("TCS-1 (Tec1)", tcs1_hits),
                    ("TCS-2 (Tec1)", tcs2_hits)]:
    fwd = sum(1 for h in hits if h.strand == "+")
    rev = sum(1 for h in hits if h.strand == "-")
    print(f"{label:20s}  +: {fwd:2d}  -: {rev:2d}  total: {len(hits)}")

### Stacked annotation tracks

Call `add_annotation_track()` or `add_annotation_track_file()` multiple times
to stack layers. Each layer gets its own colour row below the canvas.

**Track panels vs inline annotations:**
- **Track panels** (`add_annotation_track`) — suited for large sets (many hits).
  All hits in one named row; easy to compare density across the locus.
- **Inline annotations** (`add_annotation`) — suited for a small focused set.
  Labels appear directly on the canvas beside each span.

In [ ]:
fig_layers = bg_brq.plot(rows=32)

fig_layers.add_annotation_track(
    file=str(GFF3),
    filter=lambda row: row.split("\t")[2] == "gene",
    name="genes",
)
fig_layers.add_annotation_track(
    [gen.Annotation(h, name=f"PRE{h.strand}:{i}") for i, h in enumerate(pre_hits)],
    name="PRE (Ste12)",
)
fig_layers.add_annotation_track(
    [gen.Annotation(h, name=f"TCS{h.strand}:{i}") for i, h in enumerate(tcs_hits)],
    name="TCS (Tec1)",
)

print("Track panels:       ", fig_layers.annotation_tracks())
print("Inline annotations: ", fig_layers.inline_annotations())
print(f"PRE sites: {len(pre_hits)}   TCS sites: {len(tcs_hits)}")

fig_layers

### Removing and re-adding a track

`remove_annotation_track(name)` removes a single layer by name;
the widget re-renders immediately. You can call `add_annotation_track()`
again at any time to restore it.

In [ ]:
fig_layers2 = bg_brq.plot(rows=32)

fig_layers2.add_annotation_track(
    file=str(GFF3),
    filter=lambda row: row.split("\t")[2] == "gene",
    name="genes",
)
fig_layers2.add_annotation_track(
    [gen.Annotation(h, name=f"PRE{h.strand}:{i}") for i, h in enumerate(pre_hits)],
    name="PRE (Ste12)",
)
fig_layers2.add_annotation_track(
    [gen.Annotation(h, name=f"TCS{h.strand}:{i}") for i, h in enumerate(tcs_hits)],
    name="TCS (Tec1)",
)

print("Before remove:", fig_layers2.annotation_tracks())
fig_layers2.remove_annotation_track("TCS (Tec1)")
print("After remove: ", fig_layers2.annotation_tracks())

# Re-add it
fig_layers2.add_annotation_track(
    [gen.Annotation(h, name=f"TCS{h.strand}:{i}") for i, h in enumerate(tcs_hits)],
    name="TCS (Tec1)",
)
print("After re-add: ", fig_layers2.annotation_tracks())

fig_layers2

---
## Part 4 — Build the search index

**API covered:** `repo.build_index()`, `bg.build_index()`

Searching a genome graph without an index can be slow for large graphs.
Build an index once and `search()` picks it up automatically on every
subsequent call — the call is identical, just faster.

The index is stored on disk in the repository directory and persists
between sessions. Rebuild only if the graph changes.

In [ ]:
# Build a k=8 index for all block groups at once
repo.build_index(k=8)
print("Index built.")

# Identical search call — index is picked up automatically
tata_hits_indexed = bg_brq.search("tataaa")
print(f"TATA hits (full scan):  {len(tata_hits)}")
print(f"TATA hits (indexed):    {len(tata_hits_indexed)}")
assert len(tata_hits) == len(tata_hits_indexed), "Index changed the results!"

---
## Part 5 — Activator vs repressor picture

**API covered:** `fig.show(locus, color)` with explicit colours

The FLO11 promoter integrates two competing regulatory signals:
- **Activating axis** — Ste12 (PRE) + Tec1 (TCS) heterodimer drives
  pseudohyphal filamentous growth in nitrogen-poor conditions.
- **Stress axis** — Msn2/Msn4 bind STRE sites (`AGGGG`) and activate
  a broad osmostress program that competes with the filamentous growth pathway.

`fig.show(locus, color)` highlights a hit with an explicit hex colour.
Passing a colour explicitly lets you assign consistent colours to motif
classes across multiple figures rather than relying on the auto-cycle.

| Colour | Motif | TF |
|--------|-------|----|
| `#f9e2af` yellow | `TGAAACA` PRE | Ste12 (activator) |
| `#89b4fa` blue | `CATTCC/CATTCT` TCS | Tec1 (co-activator) |
| `#a6e3a1` green | `AGGGG` STRE | Msn2/Msn4 (stress competitor) |
| `#fab387` peach | `TATAAA` TATA | TBP (core promoter) |

In [ ]:
stre_hits = bg_brq.search("agggg")

MOTIFS_COLORED = [
    ("PRE (Ste12)",    "#f9e2af", pre_hits),
    ("TCS (Tec1)",     "#89b4fa", tcs_hits),
    ("STRE (Msn2/4)",  "#a6e3a1", stre_hits),
    ("TATA (core)",    "#fab387", tata_hits_indexed),
]

fig_multi_color = bg_brq.plot(rows=32)
fig_multi_color.add_annotation_track(
    file=str(GFF3),
    filter=lambda row: row.split("\t")[2] == "gene",
    name="genes",
)

for label, color, hits in MOTIFS_COLORED:
    for h in hits:
        fig_multi_color.show(h, color)
    print(f"{label:20s}  {len(hits):2d} hit(s)  [{color}]")

# Navigate to the first TATA hit
if tata_hits:
    fig_multi_color.go_to(tata_hits[0])

fig_multi_color

---
## Part 6 — Cross-strain comparison

**API covered:** `repo.search()`

`repo.search()` searches every block group and returns a list of
`(PyBlockGroup, list[GraphLocus])` pairs. This lets you compare whether a
motif is present in the reference but absent in BRQ (or vice versa),
revealing strain-specific sequence differences that may affect transcription.

A discrepancy in PRE site count between S288c and BRQ would suggest a BRQ
variant disrupts or creates a Ste12 binding site — potentially explaining
differences in flocculation behaviour between strains.

In [ ]:
hits_by_bg = repo.search("tgaaaca")

print("PRE (TGAAACA) across all block groups:")
for b, hits in hits_by_bg:
    fwd = sum(1 for h in hits if h.strand == "+")
    rev = sum(1 for h in hits if h.strand == "-")
    print(f"  {b.sample_name:15s}  total: {len(hits)}  (+: {fwd}  -: {rev})")

hits_map = {b.sample_name: hits for b, hits in hits_by_bg}

In [ ]:
fig_brq_pre = bg_brq.plot(rows=28)
fig_brq_pre.add_annotation_track(
    file=str(GFF3),
    filter=lambda row: row.split("\t")[2] == "gene",
    name="genes",
)
for h in hits_map.get('BRQ', []):
    fig_brq_pre.show(h, "#f9e2af")
print("BRQ PRE sites:")
fig_brq_pre

In [ ]:
fig_ref_pre = bg_ref.plot(rows=28)
fig_ref_pre.add_annotation_track(
    file=str(GFF3),
    filter=lambda row: row.split("\t")[2] == "gene",
    name="genes",
)
for h in hits_map.get('reference', []):
    fig_ref_pre.show(h, "#f9e2af")
print("Reference PRE sites:")
fig_ref_pre

---
## Part 7 — Removing annotations & cleanup

**API covered:** `fig.remove_annotation()`, `fig.clear_all_inline_annotations()`,
`fig.clear_all_annotations()`, `bg.clear_index()`

Three levels of annotation removal:
- `remove_annotation(name)` — remove one inline annotation by name
- `clear_all_inline_annotations()` — remove all inline annotations, keep tracks
- `clear_all_annotations()` — remove everything (inline + track panels)

In [ ]:
fig_cleanup = bg_brq.plot(rows=28)
fig_cleanup.add_annotation_track(
    file=str(GFF3),
    filter=lambda row: row.split("\t")[2] == "gene",
    name="genes",
)
for i, h in enumerate(pre_hits):
    fig_cleanup.add_annotation(gen.Annotation(h, name=f"PRE{h.strand}:{i}"))
for i, h in enumerate(tata_hits_indexed[:3]):
    fig_cleanup.add_annotation(gen.Annotation(h, name=f"TATA:{i}"))

print("Initial state:")
print("  Inline:", fig_cleanup.inline_annotations())
print("  Tracks:", fig_cleanup.annotation_tracks())

# Remove individual inline annotations by name
for i in range(len(tata_hits_indexed[:3])):
    fig_cleanup.remove_annotation(f"TATA:{i}")
print("\nAfter removing TATA inline annotations:")
print("  Inline:", fig_cleanup.inline_annotations())

# Clear all remaining inline annotations (PRE sites)
fig_cleanup.clear_all_inline_annotations()
print("\nAfter clear_all_inline_annotations():")
print("  Inline:", fig_cleanup.inline_annotations())
print("  Tracks:", fig_cleanup.annotation_tracks())  # track still present

# Clear everything including track panels
fig_cleanup.clear_all_annotations()
print("\nAfter clear_all_annotations():")
print("  Inline:", fig_cleanup.inline_annotations())
print("  Tracks:", fig_cleanup.annotation_tracks())

fig_cleanup

In [ ]:
# Clear the on-disk index for this block group
bg_brq.clear_index()
print("Index cleared. Next search() call will fall back to full scan.")

---
## Part 8 — Freeze for distribution

**How to freeze:** click the **❄** button in the widget toolbar.

This captures the current canvas as a static PNG baked into the `.ipynb`
file. Frozen widgets render on GitHub, nbviewer, and other static viewers
even without the `gen` module installed.

Navigate each live widget to the desired view, then click the freeze button.

In [ ]:
for fig in [
    fig_path,
    fig_genes,
    fig_tata_inline,
    fig_layers,
    fig_layers2,
    fig_multi_color,
    fig_brq_pre,
    fig_ref_pre,
    fig_cleanup,
]:
    break
    fig.freeze()

---
## Tips

**Minus-strand orientation** — FLO11 is encoded on the minus strand.
The PRE and TCS hits that matter for FLO11 regulation are those with
`strand == "-"` in the promoter region (positions ~4200 onward).
Plus-strand hits in the same region would regulate a gene in the
opposite orientation.

**Validating search results** — call `bg.get_locus_sequence(locus)` on any
hit to retrieve the matched sequence as a string. For `strand == "-"` hits
the return value is automatically reverse-complemented, so you always see
the sense-strand sequence.

**Index lifetime** — the index persists on disk between sessions.
Build once after import; rebuild only if the graph changes.
`bg.clear_index()` removes just that block group's index file.

**Extending the promoter window** — the FLO11 promoter is unusually large
(~3 kb). To capture all documented FRE sites, import a longer upstream
window extending ~3 kb before the FLO11 ATG.